# Experimentos de detecção de veículos

Este notebook registra ground truth, benchmarks e visualizações. A aplicação final fica em `src/`.

A imagem usada é o JPEG extraído do PDF da prova (2048×1534 px). Não registre conclusões ou métricas sem executar as células correspondentes.

In [ ]:
%matplotlib inline

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.widgets import RectangleSelector
from PIL import Image

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.evaluation.ground_truth import save_ground_truth  # noqa: E402

IMAGE_PATH = ROOT / "data/raw/drone_scene.jpg"
GROUND_TRUTH_PATH = ROOT / "data/annotations/ground_truth.json"
ANNOTATION_METHOD = "model_assisted_preannotations_manually_reviewed"
image = Image.open(IMAGE_PATH).convert("RGB")
image_width, image_height = image.size
print(f"Image: {image_width}x{image_height}")

## Anotação manual

Esta etapa é opcional: o ground truth revisado já está salvo no projeto. Para editar as caixas, execute `annotator = launch_manual_annotation()` em uma célula; o editor usa um widget interativo. Use `d` para apagar a última caixa e `s` para salvar. Antes de salvar, revise toda a rua e o estacionamento; cada veículo deve aparecer exatamente uma vez.

In [ ]:
class ManualVehicleAnnotator:
    """Interactive rectangle annotator that persists vehicle-only ground truth."""

    def __init__(self, image: Image.Image, output_path: Path) -> None:
        self.image = image
        self.output_path = output_path
        self.boxes = self._load_existing()
        self.figure, self.axis = plt.subplots(figsize=(16, 12))
        self.axis.imshow(image)
        self.axis.set_title("Drag: add vehicle | d: delete last | s: save")
        self.axis.set_axis_off()
        self.selector = RectangleSelector(
            self.axis, self._on_select, useblit=True, button=[1], interactive=False
        )
        self.figure.canvas.mpl_connect("key_press_event", self._on_key)
        self._redraw()

    def _load_existing(self) -> list[list[float]]:
        if not self.output_path.exists():
            return []
        payload = json.loads(self.output_path.read_text(encoding="utf-8"))
        return [annotation["bbox_xyxy"] for annotation in payload["annotations"]]

    def _on_select(self, start, end) -> None:
        if None in (start.xdata, start.ydata, end.xdata, end.ydata):
            return
        x1, x2 = sorted((start.xdata, end.xdata))
        y1, y2 = sorted((start.ydata, end.ydata))
        if x2 - x1 >= 3 and y2 - y1 >= 3:
            self.boxes.append([round(x1, 2), round(y1, 2), round(x2, 2), round(y2, 2)])
            self._redraw()

    def _on_key(self, event) -> None:
        if event.key == "d" and self.boxes:
            self.boxes.pop()
            self._redraw()
        elif event.key == "s":
            self.save()

    def _redraw(self) -> None:
        for patch in list(self.axis.patches):
            patch.remove()
        for index, (x1, y1, x2, y2) in enumerate(self.boxes, start=1):
            self.axis.add_patch(
                Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, ec="#00e5ff", lw=1.5)
            )
            self.axis.text(
                x1, y1, str(index), color="white", fontsize=8, bbox={"facecolor": "#004d5c"}
            )
        self.axis.set_title(f"{len(self.boxes)} vehicles | Drag: add | d: delete | s: save")
        self.figure.canvas.draw_idle()

    def save(self) -> None:
        save_ground_truth(
            self.output_path,
            IMAGE_PATH,
            self.boxes,
            annotation_method=ANNOTATION_METHOD,
            relative_to=ROOT,
        )
        print(f"Saved {len(self.boxes)} annotations to {self.output_path.relative_to(ROOT)}")


def launch_manual_annotation() -> ManualVehicleAnnotator:
    """Open the optional widget-based editor without affecting benchmark rendering."""
    get_ipython().run_line_magic("matplotlib", "widget")
    annotator = ManualVehicleAnnotator(image, GROUND_TRUTH_PATH)
    plt.show()
    return annotator


print("Ground truth editor is optional. Run annotator = launch_manual_annotation() to edit it.")

## Próximas células

Após concluir a revisão manual, este notebook carregará os detectores, executará os benchmarks controlados e apresentará as métricas e comparações visuais.

## Benchmarks reproduziveis

Execute as celulas abaixo somente apos revisar o ground truth. Cada configuracao usa a mesma imagem, o mesmo IoU de matching e tres repeticoes medidas depois de um warm-up.

In [ ]:
%matplotlib inline

import numpy as np
import pandas as pd

from src.detection.sahi_detector import SahiVehicleDetector
from src.detection.yolo_detector import YoloVehicleDetector
from src.evaluation.benchmark import benchmark_detector
from src.evaluation.ground_truth import load_ground_truth
from src.visualization.draw import draw_detections, draw_ground_truth

MODEL_DIR = ROOT / "models"
image_array = np.asarray(image)
ground_truth = load_ground_truth(GROUND_TRUTH_PATH)


def build_detector(model_name, domain, confidence=0.25, sahi_enabled=False, slice_size=512):
    model_path = MODEL_DIR / model_name
    if not model_path.exists():
        raise FileNotFoundError(
            f"Model not found: {model_path}. Run scripts/download_models.py first."
        )
    if sahi_enabled:
        return SahiVehicleDetector(
            model_path, domain, confidence, 1024, slice_size, 0.20, device="auto", merge_iou=0.50
        )
    return YoloVehicleDetector(model_path, domain, confidence, 1024, 0.70, device="auto")


def run_benchmark_table(experiments):
    rows = []
    for name, detector_options in experiments:
        result = benchmark_detector(
            name, build_detector(**detector_options), image_array, ground_truth, repetitions=3
        )
        rows.append(result.as_dict())
    return pd.DataFrame(rows)


print(f"Ground truth reviewed: {len(ground_truth)} vehicles")

In [ ]:
size_experiments = [
    ("aerial_n_normal", {"model_name": "yolo11n-obb.pt", "domain": "aerial"}),
    ("aerial_s_normal", {"model_name": "yolo11s-obb.pt", "domain": "aerial"}),
    ("aerial_m_normal", {"model_name": "yolo11m-obb.pt", "domain": "aerial"}),
]
size_results = run_benchmark_table(size_experiments)
size_results.sort_values(["f1", "median_inference_ms"], ascending=[False, True])

In [ ]:
controlled_experiments = [
    ("coco_normal", {"model_name": "yolo11n.pt", "domain": "coco"}),
    ("coco_sahi_512", {"model_name": "yolo11n.pt", "domain": "coco", "sahi_enabled": True}),
    ("aerial_normal", {"model_name": "yolo11n-obb.pt", "domain": "aerial"}),
    ("aerial_sahi_512", {"model_name": "yolo11n-obb.pt", "domain": "aerial", "sahi_enabled": True}),
]
controlled_results = run_benchmark_table(controlled_experiments)
controlled_results.sort_values("f1", ascending=False)

In [ ]:
threshold_experiments = [
    (
        f"aerial_sahi_conf_{confidence:.2f}",
        {
            "model_name": "yolo11n-obb.pt",
            "domain": "aerial",
            "confidence": confidence,
            "sahi_enabled": True,
        },
    )
    for confidence in (0.15, 0.25, 0.35, 0.50)
]
threshold_results = run_benchmark_table(threshold_experiments)

slice_experiments = [
    (
        f"aerial_sahi_slice_{slice_size}",
        {
            "model_name": "yolo11n-obb.pt",
            "domain": "aerial",
            "sahi_enabled": True,
            "slice_size": slice_size,
        },
    )
    for slice_size in (512, 640)
]
slice_results = run_benchmark_table(slice_experiments)
display(threshold_results.sort_values("f1", ascending=False))
display(slice_results.sort_values("f1", ascending=False))

In [ ]:
final_detector = build_detector("yolo11n-obb.pt", "aerial", confidence=0.25, sahi_enabled=True)
final_detections = final_detector.predict(image_array)

figure, axes = plt.subplots(1, 3, figsize=(22, 8))
for axis, title, rendered in zip(
    axes,
    ("Original", "Ground truth reviewed", "Final prediction"),
    (
        image_array,
        draw_ground_truth(image_array, ground_truth),
        draw_detections(image_array, final_detections),
    ),
    strict=True,
):
    axis.imshow(rendered)
    axis.set_title(title)
    axis.set_axis_off()
plt.tight_layout()
print(f"Final predicted count: {len(final_detections)}")

## Registro da decisao

Os resultados executados nesta imagem estao registrados em `outputs/metrics/benchmark_results.json`. A configuracao final so deve ser atualizada depois de comparar F1, recall, erro absoluto de contagem e tempo. Como o ground truth foi iniciado por pre-anotacoes do modelo aereo e revisado manualmente, os resultados sao evidencias para esta prova, nao uma estimativa de generalizacao para novas imagens.